<a href="https://colab.research.google.com/github/MuskaanPrabhakar/learning_ai/blob/main/UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ai call


In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI

In [ ]:
from google.colab import userdata

In [ ]:
client = OpenAI(
    base_url ="https://generativelanguage.googleapis.com/v1beta",
    api_key=userdata.get("GEMINI_API_KEY")
)

In [ ]:
client1 = OpenAI(
    base_url ="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY")
)

#Simple UI

In [ ]:
!pip install gradio
import gradio as gr

In [ ]:
def greet(message):
  return "hello"

In [ ]:
def talk_to_gemini(content):
  response= client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {"role": "user", "content": content}
    ],max_tokens=230
  )
  return response.choices[0].message.content

In [ ]:
gr.Interface(fn=talk_to_gemini,inputs="textbox",outputs="textbox",flagging_mode="never").launch(debug=True, auth=("musu","yoyo"))

#lil customized UI

In [ ]:
def talk_to_gpt(content):
  response= client1.chat.completions.create(
    model="openai/gpt-oss-20b:free",
    messages=[
        {"role": "user", "content": content}
    ],max_tokens=230
  )
  return response.choices[0].message.content

In [ ]:
model_input=gr.Textbox(label="your message",info="Enter a message for llm",lines=7)
model_selector=gr.Dropdown(["Gemini","GPT"],label="Select Model",value="")
message_output=gr.Textbox(label="Output",lines=7)

In [ ]:
def default(content,model_selector):
  if model_selector=="GPT":
    result= talk_to_gpt(content)
  else:
    result= talk_to_gemini(content)
  return result

In [ ]:
gr.Interface(fn=default,inputs=[model_input,model_selector],outputs=[message_output],flagging_mode="never").launch(debug=True,auth=("musu","yoyo"))

#Chat interface

In [ ]:
def talk_to_gemini(content,history):
  message=[{"role": value["role"],'content':value["content"][0]['text']} for value in history]
  message.append({"role": "user", "content": content})
  response= client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=message
  )
  return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=talk_to_gemini).launch(debug=True,auth=("musu","yoyo"))

#tools

In [ ]:
!pip install ddgs

In [ ]:
from ddgs import DDGS

In [ ]:
import json

In [ ]:
def search_web(query):
  results= DDGS().text(query,max_results=5)
  return results

In [ ]:
def add_nums(nums: list[float]):
    return sum(nums)+1

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the web for information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_nums",
            "description": "add numbers",
            "parameters": {
                "type": "object",
                "properties": {
                    "nums": {
                      "type": "array",
                            "items": {
                                     "type": "number"
                            },
                     "description": "The numbers to add."
                      }
                 },
                     "required": ["nums"]
            }
        }
    }
]

In [ ]:
contents=input()

In [ ]:
response= client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": "give tools priority"},
        {"role": "user", "content": contents}
    ],
    tools=tools
  )

In [ ]:
print(response)

In [ ]:
if(response.choices[0].message.tool_calls):
  for call in response.choices[0].message.tool_calls:
    print("The call is ",call.function.name)
    if call.function.name=="search_web":
      answer=search_web(json.loads(call.function.arguments)['query'])
      print('The answer is: ',answer)
    elif call.function.name == "add_nums":
      answer = add_nums(json.loads(call.function.arguments)['nums'])
      print(answer)


#using gpt
& gemini

In [ ]:
def talk_to_gpt(content,history):
  message=[{"role": value["role"],'content':value["content"][0]['text']} for value in history]
  message.append({"role": "user", "content": content})
  response= client1.chat.completions.create(
    model="openai/gpt-oss-20b:free",
    messages=message,
    tools=tools
  )
  if(response.choices[0].message.tool_calls):
    for call in response.choices[0].message.tool_calls:
      if call.function.name=="search_web":
        answer=search_web(json.loads(call.function.arguments)['query'])
        return str(answer) #use model-dump()..won't required json as well as str..it directly converts to str..also json dump()
      elif call.function.name == "add_nums":
        answer = add_nums(json.loads(call.function.arguments)['nums'])
        return str(answer)
  else: return response.choices[0].message.content

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the web for information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_nums",
            "description": "add numbers",
            "parameters": {
                "type": "object",
                "properties": {
                    "nums": {
                      "type": "array",
                            "items": {
                                     "type": "number"
                            },
                     "description": "The numbers to add."
                      }
                 },
                     "required": ["nums"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "scraping",
            "description": "Download the HTML content of any webpage given its URL. Use this whenever the user asks to scrape, fetch, crawl, or read a website.",
            "parameters": {
                "type": "object",
                "properties": {
                    "urlss": {
                      "type": "string",
                     "description": "website URL"
                      }
                 },
                     "required": ["urlss"]
            }
        }
    }
]

In [ ]:
def scraping(urlss):
  import requests
  return requests.get(urlss).text

In [ ]:
#better way
def talk_to_gpt(content,history):
  message=[{"role": "system", "content": "You are a helpful agent who carefully analyses the prompt and answers using tools if necessary"}]
  message.extend(
      {"role": value["role"],'content':value["content"][0]['text']} for value in history
  )
  message.append({"role": "user", "content": content})
  response= client1.chat.completions.create(
    model="openai/gpt-oss-20b:free",
    messages=message,
    tools=tools
  )
  assistant_message = response.choices[0].message
  if not assistant_message.tool_calls:
    return assistant_message.content

  message.append(assistant_message.model_dump())

  for call in response.choices[0].message.tool_calls:
    if call.function.name=="search_web":
      answer=search_web(json.loads(call.function.arguments)['query'])
    elif call.function.name == "add_nums":
      answer = add_nums(json.loads(call.function.arguments)['nums'])
    elif call.function.name == "scraping":
      answer = scraping(json.loads(call.function.arguments)['urlss'])
    else:
      answer=f"Unknown tool requested: {call.function.name}"

    message.append(
        {
            "role":"tool",
            "tool_call_id": call.id,
            "content": answer if isinstance(answer,str) else json.dumps(answer)
        }
    )

  response= client1.chat.completions.create(
  model="openai/gpt-oss-20b:free",
  messages=message,
  tools=tools
  )
  return response.choices[0].message.content


In [ ]:
#better way
def talk_to_gemini(content,history):
  message=[{"role": "system", "content": "You are a helpful agent who carefully analyses the prompt and answers using tools if necessary"}]
  message.extend(
      {"role": value["role"],'content':value["content"][0]['text']} for value in history
  )
  message.append({"role": "user", "content": content})
  response= client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=message,
    tools=tools
  )
  assistant_message = response.choices[0].message
  if not assistant_message.tool_calls:
    return assistant_message.content

  message.append({
    "role": "assistant",
    "content": assistant_message.content or "",
    "tool_calls": [
        {
            "id": tc.id,
            "type": "function",
            "function": {
                "name": tc.function.name,
                "arguments": tc.function.arguments,
            },
        }
        for tc in assistant_message.tool_calls
    ],
  })

  for call in response.choices[0].message.tool_calls:
    if call.function.name=="search_web":
      answer=search_web(json.loads(call.function.arguments)['query'])
    elif call.function.name == "add_nums":
      answer = add_nums(json.loads(call.function.arguments)['nums'])
    elif call.function.name == "scraping":
      answer = scraping(json.loads(call.function.arguments)['urlss'])
    else:
      answer=f"Unknown tool requested: {call.function.name}"

    message.append(
        {
            "role":"tool",
            "tool_call_id": call.id,
            "content": answer if isinstance(answer,str) else json.dumps(answer)
        }
    )

  response= client.chat.completions.create(
  model="gemini-2.5-flash",
  messages=message,
  tools=tools
  )
  return response.choices[0].message.content


In [ ]:
gr.ChatInterface(fn=talk_to_gemini).launch(debug=True)